In [5]:
import sys
sys.path.append("/Users/felipeformenti/dev/fformenti/nba_bets")

In [6]:
import pandas as pd
from src.config.paths import (
    REGULAR_SEASON_GAMES_PATH,
    TEAMS_CITIES_CONFERENCE_HISTORY_PROCESSED_PATH,
    LOCATIONS_DISTANCES_PATH,
    TEAMS_CITIES_LOCATIONS_HISTORY_PROCESSED_PATH,
)
from src.config.constants import TEAMS_CITIES_MAP

In [7]:
games = pd.read_csv(REGULAR_SEASON_GAMES_PATH)

/var/folders/_2/8lt451812jdgdwl3yhp1bbhh0000gn/T/ipykernel_360/703699178.py:1: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  games = pd.read_csv(REGULAR_SEASON_GAMES_PATH)


In [8]:
games.head(2)

,gameId,gameDate,gameDateOnlyStr,season,hometeamCity,hometeamName,hometeamId,awayteamCity,awayteamName,awayteamId,homeScore,awayScore,winner,overtimes,postponed,gameType,attendance,arenaId,gameLabel
0,24600052,1946-11-26 19:00:00,1946-11-26,1946/47,Philadelphia,Warriors,1610612744,Boston,Celtics,1610612738,66,54,1610612744,0,0,Regular Season,NaN,0.0,NaN
1,24600063,1946-11-30 19:00:00,1946-11-30,1946/47,New York,Knicks,1610612752,Philadelphia,Warriors,1610612744,64,60,1610612752,0,0,Regular Season,NaN,0.0,NaN


In [ ]:
distances = pd.read_csv(LOCATIONS_DISTANCES_PATH)

In [ ]:
distances.head(2)

In [ ]:
teams_cities_states = pd.read_csv(TEAMS_CITIES_LOCATIONS_HISTORY_PROCESSED_PATH)

In [ ]:
teams_cities_states[teams_cities_states["teamName"] == "Jazz"][0:2]

In [ ]:
c = games.merge(
    teams_cities_states[["teamId", "season", "state"]],
    left_on=["hometeamId", "season"],
    right_on=["teamId", "season"],
    how="left",
).drop(columns=["teamId"])

In [ ]:
c["game_location"] = c["hometeamCity"] + ", " + c["state"]

In [ ]:
games

In [ ]:
home_cols = [
    "gameId",
    "gameDate",
    "gameDateOnlyStr",
    "season",
    "hometeamId",
    "hometeamName",
    "hometeamCity",
    "game_location",
]
home_games = games[home_cols]
home_games.rename(columns={"hometeamId": "teamId"}, inplace=True)

away_cols = [
    "gameId",
    "gameDate",
    "gameDateOnlyStr",
    "season",
    "awayteamId",
    "awayteamName",
    "awayteamCity",
    "game_location",
]
away_games = games[away_cols]
away_games.rename(columns={"awayteamId": "teamId"}, inplace=True)

teams_game_dates = (
    pd.concat([home_games, away_games])
    .reset_index(drop=True)
    .rename(columns={"hometeamCity": "game_city"})
)


In [ ]:
#to do fix to grab city and state from name!!!!

In [ ]:
teams_info = pd.read_csv(TEAMS_CITIES_CONFERENCE_HISTORY_PROCESSED_PATH)

teams_game_dates = teams_game_dates.merge(
    teams_info[["teamId", "teamCity", "season"]], on=["teamId", "season"], how="inner"
)
teams_game_dates["team_city_name"] = teams_game_dates["teamCity"].replace(
    TEAMS_CITIES_MAP
)

teams_game_dates["game_city_name"] = teams_game_dates["game_city"].replace(
    TEAMS_CITIES_MAP
)

In [ ]:
season = "2024/25"
teams_game_dates = teams_game_dates[teams_game_dates["season"] == season]
teams_game_dates = teams_game_dates.sort_values(["teamId", "gameDate"])

In [ ]:
teams_game_dates[0:4]

In [ ]:
# season = "2024/25"
games_season = games.loc[games["season"] == season].copy()
season_start = games_season["gameDate"].min()
season_end = games_season["gameDate"].max()

start_date = season_start
end_date = season_end
teams_season = games_season["hometeamId"].unique()

date_range = pd.date_range(start=start_date, end=end_date)
full_calendar_teams = pd.MultiIndex.from_product(
    [date_range, teams_season], names=["gameDate", "teamId"]
).to_frame(index=False)
full_calendar_teams["gameDateOnlyStr"] = full_calendar_teams["gameDate"].dt.strftime(
    "%Y-%m-%d"
)

full_calendar_teams.drop(columns=["gameDate"], inplace=True)
full_calendar_teams = full_calendar_teams.merge(
    teams_game_dates, on=["teamId", "gameDateOnlyStr"], how="left"
)
full_calendar_teams = full_calendar_teams.sort_values(["teamId", "gameDateOnlyStr"]).reset_index(drop=True)

In [ ]:
full_calendar_teams["team_city_name"].fillna(method="ffill", inplace=True)

In [ ]:
full_calendar_teams["current_city"] = full_calendar_teams["game_city_name"]

full_calendar_teams["next_game_city_name"] = full_calendar_teams.groupby(["teamId"])[
    "game_city_name"
].shift(-1)

In [ ]:
def conditional_backfill_group(group):
    group["current_city"] = group["current_city"].mask(
        group["current_city"].isna()
        & (group["next_game_city_name"].bfill() == group["team_city_name"]),
        group["next_game_city_name"].bfill(),
    )
    return group


full_calendar_teams = full_calendar_teams.groupby("teamId", group_keys=False).apply(
    conditional_backfill_group
)

# to do:  repensar se um
full_calendar_teams["current_city"] = full_calendar_teams.groupby("teamId")[
    "current_city"
].fillna(method="ffill")


full_calendar_teams["team_city_name"] = full_calendar_teams.groupby("teamId")[
    "team_city_name"
].fillna(method="bfill")

In [ ]:
full_calendar_teams


In [ ]:
full_calendar_teams.drop(columns=["next_game_city_name"], inplace=True)
full_calendar_teams[full_calendar_teams["teamCity"] == "New York"]


In [ ]:
full_calendar_teams["previous_city"] = full_calendar_teams.groupby(["teamId"])[
    "current_city"
].shift(1)


In [ ]:
def if_na_select_column(group):
    group["previous_city"] = group["previous_city"].mask(
        group["previous_city"].isna(),
        group["team_city_name"],
    )
    return group


full_calendar_teams = full_calendar_teams.groupby("teamId", group_keys=False).apply(
    if_na_select_column
)

full_calendar_teams["current_city"] = full_calendar_teams.groupby("teamId")[
    "current_city"
].fillna(method="bfill")


In [ ]:
distances = pd.read_csv(RAW_DISTANCES_PATH)

aux_distances = distances.rename(columns={"city1": "city2", "city2": "city1"})
distances = pd.concat([distances, aux_distances]).reset_index(drop=True)

# teams_game_distances = teams_game_dates.merge(
#     distances,
#     left_on=["team_city_name", "game_city_name"],
#     right_on=["city1", "city2"],
#     how="left",
# ).drop(columns=["city1", "city2"])

# # fill distance column with 0 when na and turn columns to integer
# teams_game_distances["distance"] = (
#     teams_game_distances["distance"].fillna(0).astype(int)
# )


In [ ]:
full_calendar_teams = full_calendar_teams.merge(
    distances,
    left_on=["current_city", "previous_city"],
    right_on=["city1", "city2"],
    how="left",
)


In [ ]:
full_calendar_teams["distance"] = full_calendar_teams["distance"].fillna(0).astype(int)


In [ ]:
full_calendar_teams.drop(columns=["city1", "city2"], inplace=True)

In [ ]:
full_calendar_teams["distance_traveled_L1"] = (
    full_calendar_teams.groupby(["teamId"])["distance"]
    .rolling(window=1, min_periods=1)
    .mean()
    .reset_index(0, drop=True)
)


In [ ]:
full_calendar_teams[0:50]
full_calendar_teams[["teamId", "season", "gameDateOnlyStr", "distance"]]

In [ ]:
(0.50 - 0.45) / 0.45

In [ ]:
5/4.5

In [ ]:
def if_na_select_column(group):
    group["next_game_city_name"] = group["next_game_city_name"].mask(
        group["next_game_city_name"].isna(),
        group["previous_game_city_name"],
    )
    return group

full_calendar_teams = full_calendar_teams.groupby("teamId", group_keys=False).apply(
    if_na_select_column
)

In [ ]:
full_calendar_teams["next_game_city_name"].fillna(method="ffill", inplace=True)

In [ ]:

# full_calendar_teams.drop(columns=["previous_game_city_name"], inplace=True)
# full_calendar_teams.rename(columns={"next_game_city_name": "last_stop"}, inplace=True)
# full_calendar_teams[~full_calendar_teams[["gameId"]].isna().any(axis=1)][0:50]


In [ ]:
teams_game_dates["last_game_city_name"] = teams_game_dates.groupby(
    ["teamId"]
)["game_city_name"].shift(1)

In [ ]:
teams_game_dates[0:50]


In [ ]:
distances = pd.read_csv(RAW_DISTANCES_PATH)

aux_distances = distances.rename(columns={"city1": "city2", "city2": "city1"})
distances = pd.concat([distances, aux_distances]).reset_index(drop=True)

teams_game_distances = teams_game_dates.merge(
    distances,
    left_on=["team_city_name", "game_city_name"],
    right_on=["city1", "city2"],
    how="left",
).drop(columns=["city1", "city2"])

# fill distance column with 0 when na and turn columns to integer
teams_game_distances["distance"] = (
    teams_game_distances["distance"].fillna(0).astype(int)
)

teams_game_distances[teams_game_distances["gameId"] == 22500786]

In [ ]:
# teams_game_distances.loc[
#     (teams_game_distances["season"] == "2024/25") & ((teams_game_distances["team_city_name"] == "San Francisco") | (teams_game_distances["game_city_name"] == "San Francisco"))
# ][0:50]

# team_city_name = "Portland"
team_city_name = "San Francisco"
teams_game_distances.loc[
    (teams_game_distances["season"] == "2024/25")
    & (teams_game_distances["team_city_name"] == team_city_name)
].sort_values(by="gameDate")[0:50]
# teams_game_distances[teams_game_distances["gameId"] == 22500786]


In [ ]:
teams_ids_cities = games[["hometeamId", "hometeamCity"]].drop_duplicates()
teams_ids_cities.rename(columns={"hometeamId": "teamId", "hometeamCity": "teamCity"}, inplace=True)
teams_game_dates = teams_game_dates.merge(teams_ids_cities, on="teamId")

In [ ]:
teams_ids_cities.sort_values("teamId", inplace=False)

In [ ]:
# games[games["hometeamId"] == 1610612746]


In [ ]:
# raw_games["awayteamCity"] = raw_games["awayteamCity"].replace(normalize_citi_names)

In [ ]:
# raw_games[["hometeamId", "hometeamCity"]].drop_duplicates()

In [ ]:
# teams_game_dates[teams_game_dates["teamId"] == 1610612745]
# teams_game_dates[teams_game_dates["season"] == season]